# 1 — Setup & Data Loading

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import OneHotEncoder,LabelEncoder,StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

In [ ]:
try:
    import imblearn
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'imbalanced-learn', '--quiet'])
    import imblearn
print('imbalanced-learn', imblearn.__version__)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
try:
    df= pd.read_csv('/content/drive/MyDrive/NTI/task/healthcare-dataset-stroke-data.csv')
except FileNotFoundError:
    df= pd.read_csv('healthcare-dataset-stroke-data.csv')


## 2 — Exploratory Data Analysis (Raw)

In [ ]:
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
numcol=['age','avg_glucose_level','bmi']
catcol=['gender','ever_married','work_type','Residence_type','smoking_status','heart_disease','stroke']

In [ ]:
for col in catcol:
  df[col].value_counts().plot(kind='bar')
  plt.show()

In [ ]:
corr =df[numcol].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')

In [ ]:

for col in numcol:
  sns.boxplot(data=df,x=col,hue="stroke")
  plt.show()

In [ ]:
for col in catcol:
  sns.countplot(data=df,x=col,hue="stroke")
  plt.show()

## 3 — Data Cleaning

In [ ]:
df.drop(columns=['id'], inplace=True)

In [ ]:
df['bmi'] = df['bmi'].fillna(df['bmi'].median())


In [ ]:
print('gender before:', df['gender'].value_counts().to_dict())
df.drop(df[df['gender'] == 'Other'].index, inplace=True)
print('gender after:', df['gender'].value_counts().to_dict())


In [ ]:
df.loc[(df['smoking_status'] == 'Unknown') & (df['age'] < 15), 'smoking_status'] = 'never smoked'
mode_value = df.loc[df['smoking_status'] != 'Unknown', 'smoking_status'].mode()[0]
df.loc[df['smoking_status'] == 'Unknown', 'smoking_status'] = mode_value

In [ ]:
df['smoking_status'].unique()

In [ ]:
print('smoking_status after cleaning:', df['smoking_status'].value_counts().to_dict())
print('Unknown remaining:', (df['smoking_status']=='Unknown').sum())
assert (df['smoking_status']=='Unknown').sum()==0, 'Unknown still present!'
df.info()


## 4 — Feature Engineering

In [ ]:
df['log_age'] = np.log1p(df['age'])
df['log_bmi'] = np.log1p(df['bmi'])
df['log_glucose'] = np.log1p(df['avg_glucose_level'])

In [ ]:
df['bmi_glucose_interaction'] = df['log_bmi'] * df['log_age']
df['high_risk_elderly'] = ((df['age'] > 65) & ((df['hypertension'] == 1) | (df['heart_disease'] == 1))).astype(int)
df['risk_factor_count'] = df['hypertension'] + df['heart_disease'] + (df['age'] > 65).astype(int)

In [ ]:
df.drop(columns=['age', 'bmi', 'avg_glucose_level'], inplace=True)

In [ ]:
print(df.columns.tolist())
print(df[['log_age','log_bmi','log_glucose','bmi_glucose_interaction','high_risk_elderly','risk_factor_count']].head())


## 5 — Train/Test Split & Preprocessing

In [ ]:
x = df.drop('stroke', axis=1)

y = df['stroke']

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", x_train.shape)
print("Testing data:", x_test.shape)

In [ ]:
cat_feture=['gender',
 'ever_married',
 'work_type',
 'Residence_type',
 'smoking_status']
num_feture=['log_age',
 'log_bmi',
 'log_glucose',
 'bmi_glucose_interaction',
 'high_risk_elderly',
 'risk_factor_count']

In [ ]:

preprocessor = ColumnTransformer(
    transformers=[
        ('robust', StandardScaler(), num_feture),
        ('onehot', OneHotEncoder(handle_unknown='ignore'),cat_feture)
    ],
    remainder='passthrough'
)

x_train_processed = preprocessor.fit_transform(x_train)
x_test_processed = preprocessor.transform(x_test)

In [ ]:
x_train_processed=pd.DataFrame(x_train_processed,columns=preprocessor.get_feature_names_out())
x_test_processed=pd.DataFrame(x_test_processed,columns=preprocessor.get_feature_names_out())

In [ ]:
x_train_processed

In [ ]:
print('Train:', x_train_processed.shape, ' Test:', x_test_processed.shape)
print(list(preprocessor.get_feature_names_out())[:12])


## 6 — Baseline Modeling (class_weight='balanced')

In [ ]:
model= LogisticRegression()
model.fit(x_train_processed,y_train)
y_pred_train=model.predict(x_train_processed)
y_pred_test=model.predict(x_test_processed)

print("Training Accuracy:", accuracy_score(y_train, y_pred_train))
print("Testing Accuracy:", accuracy_score(y_test, y_pred_test))
print("Classification Report:\n", classification_report(y_test, y_pred_test))
print("R2 Score:", model.score(x_test_processed, y_test))

In [ ]:
cm_logistic = confusion_matrix(y_test, y_pred_test)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm_logistic,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.title('Logistic Regression - Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')

plt.show()

In [ ]:
model= LogisticRegression(class_weight='balanced')
model.fit(x_train_processed,y_train)
y_pred_train=model.predict(x_train_processed)
y_pred_test=model.predict(x_test_processed)

print("Training Accuracy:", accuracy_score(y_train, y_pred_train))
print("Testing Accuracy:", accuracy_score(y_test, y_pred_test))
print("Classification Report:\n", classification_report(y_test, y_pred_test))
print("R2 Score:", model.score(x_test_processed, y_test))

In [ ]:
cm_logistic = confusion_matrix(y_test, y_pred_test)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm_logistic,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.title('Logistic Regression - Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')

plt.show()

In [ ]:
model_svc2 = SVC(kernel='linear',C=1,class_weight='balanced')
model_svc2.fit(x_train_processed,y_train)
y_pred_train_svc2=model_svc2.predict(x_train_processed)
y_pred_test_svc2=model_svc2.predict(x_test_processed)

print("Training Accuracy:", accuracy_score(y_train, y_pred_train_svc2))
print("Testing Accuracy:", accuracy_score(y_test, y_pred_test_svc2))
print("Classification Report:\n", classification_report(y_test, y_pred_test_svc2))
print("R2 Score:", model_svc2.score(x_test_processed, y_test))

In [ ]:
cm_logistic = confusion_matrix(y_test, y_pred_test_svc2)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm_logistic,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.title('SVC (linear, C=1, balanced) - Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')

plt.show()

In [ ]:
model_svc3 = SVC(kernel='poly',degree=4,C=1, gamma='scale',class_weight='balanced')
model_svc3.fit(x_train_processed,y_train)
y_pred_train_svc3=model_svc3.predict(x_train_processed)
y_pred_test_svc3=model_svc3.predict(x_test_processed)


print("Training Accuracy:", accuracy_score(y_train, y_pred_train_svc3))
print("Testing Accuracy:", accuracy_score(y_test, y_pred_test_svc3))
print("Classification Report:\n", classification_report(y_test, y_pred_test_svc3))
print("R2 Score:", model_svc3.score(x_test_processed, y_test))

In [ ]:
cm_logistic = confusion_matrix(y_test, y_pred_test_svc3)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm_logistic,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.title('SVC (poly degree=4, C=1, balanced) - Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')

plt.show()


In [ ]:
model_svc3 = SVC(kernel='rbf',C=1, gamma='auto',class_weight='balanced')
model_svc3.fit(x_train_processed,y_train)
y_pred_train_svc3=model_svc3.predict(x_train_processed)
y_pred_test_svc3=model_svc3.predict(x_test_processed)


print("Training Accuracy:", accuracy_score(y_train, y_pred_train_svc3))
print("Testing Accuracy:", accuracy_score(y_test, y_pred_test_svc3))
print("Classification Report:\n", classification_report(y_test, y_pred_test_svc3))
print("R2 Score:", model_svc3.score(x_test_processed, y_test))

In [ ]:
cm_logistic = confusion_matrix(y_test, y_pred_test_svc3)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm_logistic,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.title('SVC (rbf, C=1, balanced) - Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')

plt.show()


In [ ]:
model_svc2 = SVC(kernel='linear',C=10,class_weight='balanced')
model_svc2.fit(x_train_processed,y_train)
y_pred_train_svc2=model_svc2.predict(x_train_processed)
y_pred_test_svc2=model_svc2.predict(x_test_processed)

print("Training Accuracy:", accuracy_score(y_train, y_pred_train_svc2))
print("Testing Accuracy:", accuracy_score(y_test, y_pred_test_svc2))
print("Classification Report:\n", classification_report(y_test, y_pred_test_svc2))
print("R2 Score:", model_svc2.score(x_test_processed, y_test))

In [ ]:
cm_logistic = confusion_matrix(y_test, y_pred_test_svc2)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm_logistic,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.title('SVC (linear, C=10, balanced) - Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')

plt.show()


In [ ]:
model_svc3 = SVC(kernel='sigmoid',C=1, gamma='auto',class_weight='balanced')
model_svc3.fit(x_train_processed,y_train)
y_pred_train_svc3=model_svc3.predict(x_train_processed)
y_pred_test_svc3=model_svc3.predict(x_test_processed)


print("Training Accuracy:", accuracy_score(y_train, y_pred_train_svc3))
print("Testing Accuracy:", accuracy_score(y_test, y_pred_test_svc3))
print("Classification Report:\n", classification_report(y_test, y_pred_test_svc3))
print("R2 Score:", model_svc3.score(x_test_processed, y_test))

In [ ]:
cm_logistic = confusion_matrix(y_test, y_pred_test_svc3)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm_logistic,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.title('SVC (sigmoid, C=1, balanced) - Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')

plt.show()

## 7 — Handling Class Imbalance with SMOTE

In [ ]:
from imblearn.over_sampling import SMOTE

print("Before SMOTE:", pd.Series(y_train).value_counts().to_dict())
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(x_train_processed, y_train)
print("After SMOTE:", pd.Series(y_train_resampled).value_counts().to_dict())


In [ ]:
model= LogisticRegression()
model.fit(X_train_resampled,y_train_resampled)
y_pred_train=model.predict(X_train_resampled)
y_pred_test=model.predict(x_test_processed)

print("Training Accuracy:", accuracy_score(y_train_resampled, y_pred_train))
print("Testing Accuracy:", accuracy_score(y_test, y_pred_test))
print("Classification Report:\n", classification_report(y_test, y_pred_test))
print("R2 Score:", model.score(x_test_processed, y_test))

In [ ]:
cm_logistic = confusion_matrix(y_test, y_pred_test)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm_logistic,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.title('Logistic Regression + SMOTE - Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')

plt.show()

### 7.1 — Hyperparameter Tuning (GridSearchCV)

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
grid_pram={
    "C":[0.01,0.1,1,10,100],
    'gamma':['auto','scale'],
    'kernel':['linear','poly','rbf','sigmoid']
}
grid=GridSearchCV(SVC(),param_grid=grid_pram,verbose=5)
grid.fit(X_train_resampled,y_train_resampled)

In [ ]:
print('Best params:', grid.best_params_)
print('Best score:', grid.best_score_)


In [ ]:
model_svc3 = SVC(kernel='rbf',C=100, gamma='auto',class_weight='balanced')
model_svc3.fit(X_train_resampled,y_train_resampled)
y_pred_train_svc3=model_svc3.predict(X_train_resampled)
y_pred_test_svc3=model_svc3.predict(x_test_processed)


print("Training Accuracy:", accuracy_score(y_train_resampled, y_pred_train_svc3))
print("Testing Accuracy:", accuracy_score(y_test, y_pred_test_svc3))
print("Classification Report:\n", classification_report(y_test, y_pred_test_svc3))
print("R2 Score:", model_svc3.score(x_test_processed, y_test))

In [ ]:
cm_logistic = confusion_matrix(y_test, y_pred_test_svc3)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm_logistic,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.title('SVC (rbf, C=100, balanced) + SMOTE - Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')

plt.show()


In [ ]:
model_svc3 = SVC(kernel='poly',degree=3,C=100, gamma='auto',class_weight='balanced')
model_svc3.fit(X_train_resampled,y_train_resampled)
y_pred_train_svc3=model_svc3.predict(X_train_resampled)
y_pred_test_svc3=model_svc3.predict(x_test_processed)


print("Training Accuracy:", accuracy_score(y_train_resampled, y_pred_train_svc3))
print("Testing Accuracy:", accuracy_score(y_test, y_pred_test_svc3))
print("Classification Report:\n", classification_report(y_test, y_pred_test_svc3))
print("R2 Score:", model_svc3.score(x_test_processed, y_test))

In [ ]:
cm_logistic = confusion_matrix(y_test, y_pred_test_svc3)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm_logistic,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.title('SVC (poly degree=3, C=100, balanced) + SMOTE - Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')

plt.show()
